# 토큰 사용량(Token Usage) 추적 실습

LLM 호출 비용은 보통 입력/출력 토큰 수에 비례해서 청구된다. `get_usage_metadata_callback()`을 이용하면, `with` 블록 안에서 이루어진 호출들의 토큰 사용량을 모델별로 자동 집계해서 확인할 수 있다.

## 1. 환경변수 로드

`.env`에 저장된 `OPENAI_API_KEY`를 불러온다.

In [1]:
from dotenv import load_dotenv

# .env 파일의 OPENAI_API_KEY 등 환경변수를 불러온다.
load_dotenv()

True

## 2. 콜백과 모델 준비

`get_usage_metadata_callback`은 `with` 블록으로 사용하는 컨텍스트 매니저로, 그 블록 안에서 호출된 LLM들의 토큰 사용량을 자동으로 집계해준다.

In [4]:
from langchain_core.callbacks import get_usage_metadata_callback
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-5.6-luna")

## 3. 호출 1번의 토큰 사용량 확인

`with get_usage_metadata_callback() as cb:` 블록 안에서 `llm.invoke()`를 호출하면, 그 호출에 사용된 토큰 수가 `cb.usage_metadata`에 모델 이름을 key로 하는 dict 형태로 쌓인다. (`input_tokens`, `output_tokens`, `total_tokens` 등을 포함)

In [6]:
with get_usage_metadata_callback() as cb:
    result = llm.invoke("대한민국의 수도는 어디야?")

# 모델 이름을 key로, 입력/출력/총 토큰 수 등을 값으로 갖는 dict가 담긴다.
print(cb.usage_metadata)

{'gpt-5.6-luna': {'input_tokens': 14, 'output_tokens': 16, 'total_tokens': 30, 'input_token_details': {'audio': 0, 'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}}


## 4. 여러 번 호출한 토큰 사용량 합산하기

같은 `with` 블록 안에서 `invoke()`를 여러 번 호출하면, 그 블록 안에서 이루어진 모든 호출의 토큰 사용량이 `cb.usage_metadata`에 누적된다. `usage_metadata`는 `{모델명: {input_tokens, output_tokens, ...}}` 형태의 dict이므로, `.values()`를 순회하면서 각 모델의 토큰 수를 더하면 전체 합계를 구할 수 있다(여기서는 모델이 하나뿐이라 사실상 그 모델의 두 번 호출분 합계가 된다).

In [7]:
with get_usage_metadata_callback() as cb:
    # 같은 블록 안에서 두 번 호출 -> 두 호출의 토큰 사용량이 함께 누적된다.
    result = llm.invoke("대한민국의 수도는 어디야?")
    result = llm.invoke("대한민국의 수도는 어디야?")

usage = cb.usage_metadata

# usage는 {모델명: {input_tokens, output_tokens, total_tokens, ...}} 형태의 dict.
# 여러 모델을 섞어 썼다면 .values()로 순회하며 각 모델의 사용량을 합산할 수 있다.
input_tokens = sum(
    data["input_tokens"]
    for data in usage.values()
)

output_tokens = sum(
    data["output_tokens"]
    for data in usage.values()
)

total_tokens = sum(
    data["total_tokens"]
    for data in usage.values()
)

print(f"총 사용된 토큰수: {total_tokens}")
print(f"입력 토큰수: {input_tokens}")
print(f"출력 토큰수: {output_tokens}")

총 사용된 토큰수: 60
입력 토큰수: 28
출력 토큰수: 32
